In [ ]:
from sqlalchemy.testing.suite.test_reflection import metadata
%pip install uv
# fastembed 및 평가용 라이브러리 설치. dependency conflict 회피를 위해 force-reinstall 또는 호환 버전 명시 고려 가능
%pip install fastembed tqdm transformers torch huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from huggingface_hub import login
from dotenv import find_dotenv, load_dotenv
import os

# 1. 환경 변수 로드
load_dotenv(find_dotenv())
os.environ['HF_HOME'] = 'E:/huggingface_cache'
login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [14]:
from qdrant_client.http.models import VectorParams, Distance, SparseVectorParams, SparseIndexParams
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from fastembed import SparseTextEmbedding, SparseEmbedding
from langchain_upstage import ChatUpstage
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from qdrant_client import QdrantClient
import mysql.connector
import os
import random
from tqdm import tqdm
import torch

print(torch.cuda.is_available())


# 2. DB 설정
db_config = {
    'host': os.getenv('DB_HOST'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_NAME')
}

qdrant_url = os.getenv('QDRANT_URL')

# 모델 목록 정의
# (사용자 요청: naver/splade-v3, yjoonjang/splade-ko-v1, Qdrant/bm25)
# FastEmbed 사용 시, 모델명이 FastEmbed에서 지원하는 이름이어야 하거나, 
# 지원하지 않는 경우 직접 Transformers 등을 활용한 호환 Wrapper가 필요할 수 있음.
# 여기서는 FastEmbedSparse의 인터페이스를 최대한 활용.
MODELS_TO_COMPARE = [
    
    {"name": "Qdrant/bm25", "type": "fastembed"},
    #{"name": "naver/splade-v3", "type": "custom_splade"}, 
    {"name": "yjoonjang/splade-ko-v1", "type": "custom_splade"},
]

False


In [10]:
import torch
from typing import List, Dict
from transformers import AutoModelForMaskedLM, AutoTokenizer
from langchain_core.embeddings import Embeddings
from qdrant_client.http import models as rest_models

class KoreanSpladeSparseEmbedding(Embeddings):
    """
    yjoonjang/splade-ko-v1 모델을 지원하기 위한 커스텀 임베딩 클래스
    """
    def __init__(self, model_name: str = "yjoonjang/splade-ko-v1", device: str = None):
        print(f"Loading Korean SPLADE model: {model_name}...")
        
        # GPU 사용 가능 여부 자동 체크
        if not device:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            self.device = device
            
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForMaskedLM.from_pretrained(model_name).to(self.device)
        self.model.eval() # 평가 모드 설정
        print(f"Model loaded on {self.device}.")

    def _compute_splade(self, texts: List[str]) -> List[rest_models.SparseVector]:
        # 1. 토크나이징
        tokens = self.tokenizer(
            texts, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=512
        ).to(self.device)

        with torch.no_grad():
            # 2. 모델 추론 (Logits 추출)
            output = self.model(**tokens)
            logits = output.logits # [Batch, Seq_Len, Vocab_Size]

            # 3. SPLADE 공식 적용: log(1 + ReLU(logits))
            # 릴루(ReLU)를 통해 음수 제거 -> 로그 스케일링
            values = torch.log(1 + torch.relu(logits))

            # 4. Max Pooling (문장 내에서 각 단어의 최대 영향력 추출)
            # Attention Mask를 적용하여 패딩 토큰 무시
            attention_mask = tokens["attention_mask"].unsqueeze(-1)
            values = values * attention_mask
            
            # 문장 길이(Sequence) 축에 대해 Max 값을 취함 -> [Batch, Vocab_Size]
            sparse_vecs, _ = torch.max(values, dim=1)

        # 5. Qdrant 포맷으로 변환 (0이 아닌 값만 추출)
        results = []
        sparse_vecs = sparse_vecs.cpu().to_dense().numpy() # 넘파이로 변환

        for vec in sparse_vecs:
            indices = vec.nonzero()[0] # 값이 0이 아닌 인덱스들
            values = vec[indices]      # 해당 인덱스의 값들
            
            # Qdrant SparseVector 객체 생성
            results.append(rest_models.SparseVector(
                indices=indices.tolist(), 
                values=values.tolist()
            ))
            
        return results

    def embed_documents(self, texts: List[str]) -> List[rest_models.SparseVector]:
        return self._compute_splade(texts)

    def embed_query(self, text: str) -> rest_models.SparseVector:
        return self._compute_splade([text])[0]

In [11]:
# 4. 데이터 로드 (SQL JOIN)
all_docs = []

try:
    connection = mysql.connector.connect(**db_config)
    cursor = connection.cursor(dictionary=True)

    # attractions와 tags를 JOIN
    # GROUP_CONCAT으로 태그들을 합쳐서 가져옵니다.
    query = """
        SELECT 
            a.no, a.title, a.overview, a.latitude, a.longitude,
            GROUP_CONCAT(t.name SEPARATOR ', ') as tag_names
        FROM attractions a
        LEFT JOIN attraction_tag at ON a.no = at.attraction_id
        LEFT JOIN tags t ON at.tag_id = t.id
        WHERE a.overview IS NOT NULL AND a.overview != ''
        GROUP BY a.no
        LIMIT 500
    """
    cursor.execute(query)
    rows = cursor.fetchall()

    for row in rows:
        if float(row["latitude"]) != 0 and float(row["longitude"]) != 0:
            # 태그가 있으면 overview 뒤에 덧붙여서 풍부한 검색이 되도록 함
            content = row["overview"]
            if row["tag_names"]:
                content += f"\n태그: {row['tag_names']}"
                
            doc = Document(
                page_content=content,
                metadata={
                    "id": row["no"],
                    "title": row["title"],
                    "tags": row["tag_names"] if row["tag_names"] else ""
                }
            )
            all_docs.append(doc)
    
    print(f"총 {len(all_docs)}개의 문서 로드 완료 (Tags 포함).")

except Exception as e:
    print(f"Error: {e}")
finally:
    if 'connection' in locals() and connection.is_connected():
        cursor.close()
        connection.close()

총 500개의 문서 로드 완료 (Tags 포함).


In [12]:
# 5. 평가용 쿼리 생성 (Natural Only)
qa_pairs = []
llm = ChatUpstage(model="solar-1-mini-chat")

prompt_template = PromptTemplate.from_template(
    """다음 관광지 정보를 보고, 사용자가 이 장소를 찾을 때 검색창에 입력할만한 '자연어 질문' 1개를 만들어주세요.
    질문은 해당 장소의 특징(태그 등)을 묘사하며 찾는 형태여야 합니다.
    
    형식: [질문]
    예시: 서울에서 야경이 예쁘고 데이트하기 좋은 곳은 어디인가요?
    
    관광지 정보:
    {context}
    """
)

chain = prompt_template | llm | StrOutputParser()

sample_docs = random.sample(all_docs, min(40, len(all_docs))) # 20 -> 40개로 증가

print("자연어 쿼리 생성 중...")
for doc in tqdm(sample_docs):
    try:
        res = chain.invoke({"context": doc.page_content[:1000]}).strip()
        # 혹시 모를 대괄호 제거 등에 대한 후처리
        query = res.replace("[", "").replace("]", "").strip()
        
        qa_pairs.append({
            "query": query,
            "ground_truth_id": doc.metadata["id"],
            "type": "natural"
        })
    except:
        continue
        
print(f"총 {len(qa_pairs)}개의 평가 쿼리 생성 (Natural Only)")

자연어 쿼리 생성 중...


100%|██████████| 40/40 [00:22<00:00,  1.78it/s]

총 40개의 평가 쿼리 생성 (Natural Only)


In [16]:
# 6. 모델별 비교 평가 실행
results_summary = {}

for model_info in MODELS_TO_COMPARE:
    model_name = model_info["name"]
    print(f"\n=== Evaluating Model: {model_name} ===")
    
    # 1) Sparse Encoder 초기화
    if model_info["type"] == "fastembed":
        # fastembed 지원 모델 (fallback)
        try:
            sparse_embeddings = FastEmbedSparse(model_name=model_name)
        except Exception as e:
            print(f"FastEmbed 로드 실패 ({model_name}): {e}. Skip.")
            continue
            
    else:
        # Custom Wrapper (Transformers)
        sparse_embeddings = KoreanSpladeSparseEmbedding(model_name=model_name)
        
    # 2) Qdrant Collection 준비 (Memory)
    client = QdrantClient(":memory:")
    collection_name = f"bench_{model_name.replace('/', '_')}"
    
    # 컬렉션이 없으면 생성 (Sparse Vector Config 필수)
    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config={}, # Dense Vector 미사용
            sparse_vectors_config={
                "langchain-sparse": SparseVectorParams(
                    index=SparseIndexParams(
                        on_disk=False,
                    )
                )
            }
        )
    
    # 3) Vector Store 생성 & 인덱싱
    vector_store = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=None, # Dense 사용 안함 (Pure Sparse Test)
        sparse_embedding=sparse_embeddings,
        sparse_vector_name="langchain-sparse", # 명시적 이름 지정
        retrieval_mode=RetrievalMode.SPARSE
    )
    
    print("Indexing...")
    vector_store.add_documents(all_docs)
    
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 5,
            # score_threshold는 필요 시 주석 해제하여 사용
            # "score_threshold": 0.01 
        }
    )
    
    # 4) 평가 (Hit Rate, MRR)
    hits = 0
    mrr_sum = 0
    k = 5
    
    for item in tqdm(qa_pairs):
        query = item["query"]
        target_id = item["ground_truth_id"]
        
        try:
            search_res = retriever.invoke(query)
            found = False
            for rank, doc in enumerate(search_res):
                if doc.metadata.get("id") == target_id:
                    hits += 1
                    mrr_sum += 1.0 / (rank + 1)
                    found = True
                    break
        except Exception as e:
            print(f"Search Error: {e}")

    hit_rate = hits / len(qa_pairs)
    mrr = mrr_sum / len(qa_pairs)
    
    results_summary[model_name] = {"Hit Rate@5": hit_rate, "MRR@5": mrr}
    print(f"Result: {results_summary[model_name]}")

print("\n=== 최종 비교 결과 ===")
for m, scores in results_summary.items():
    print(f"{m}: {scores}")


=== Evaluating Model: Qdrant/bm25 ===
Indexing...


100%|██████████| 40/40 [00:00<00:00, 66.35it/s]


Result: {'Hit Rate@5': 0.4, 'MRR@5': 0.27041666666666664}

=== Evaluating Model: yjoonjang/splade-ko-v1 ===
Loading Korean SPLADE model: yjoonjang/splade-ko-v1...
Model loaded on cpu.
Indexing...


KeyboardInterrupt: 